# Introduction to JAX

JAX provides numerical arrays similar to NumPy arrays, adding automatic differentiation and function transformations. Here, we approach it as a computational tool for mathematical machines.

By the end of this notebook, you should be able to:

- create and manipulate JAX arrays;
- distinguish componentwise and matrix products;
- functionally update an immutable array;
- compute a gradient, Hessian or Jacobian matrix;
- use `vmap`, `jit` and random keys;
- take a few gradient steps for an affine machine.

## Exercises

1. [Euclidean and Frobenius inner products](#Exercise-1)
2. [Functional updates](#Exercise-2)
3. [Directional derivative and Hessian](#Exercise-3)
4. [Identifying an affine machine](#Exercise-4)

## 1. Arrays and matrix calculations

By convention, we import `jax.numpy` as `jnp`. Most elementary operations use the same syntax as NumPy. Here, we enable 64-bit floating-point numbers, useful for scientific computing. This configuration must be set at the start of the notebook.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp

print(f"JAX {jax.__version__}")

An array has, in particular, a shape (`shape`) and a numerical type (`dtype`). The `*` operator acts componentwise; `@` represents matrix composition.

In [ ]:
x = jnp.array([1.0, 2.0, -1.0])
z = jnp.array([2.0, -1.0, 3.0])
A = jnp.array([[1.0, 2.0, 0.0], [0.0, -1.0, 3.0]])

print(f"{x.shape=} {x.dtype=}")
print(f"{A.shape=} {A.dtype=}")
print(f"x * z = {x * z}")             # Hadamard product
print(f"A @ x = {A @ x}")             # matrix-vector product
print(f"A.T.shape = {A.T.shape}")      # transpose

### Exercise 1

**Euclidean and Frobenius inner products.**

Choose $y\in\mathbb R^2$ and verify numerically the identity

$$
\langle y,Ax\rangle=\langle yx^\top,A\rangle_F.
$$

The right-hand side can be computed with `jnp.sum((y[:, None] * x[None, :]) * A)`. Explain the shapes of the intermediate arrays.

In [ ]:
# Complete this section.

## 2. Immutable arrays

Unlike NumPy arrays, JAX arrays are immutable. An instruction such as `B[0, 1] = 2` therefore raises an error. The `.at` notation constructs a new array and leaves the old one unchanged.

In [ ]:
B = jnp.zeros((3, 3))
B1 = B.at[0, 1].set(2.0)
B2 = B1.at[:, 0].add(jnp.array([1.0, 2.0, 3.0]))

print("B =\n", B)
print("B1 =\n", B1)
print("B2 =\n", B2)

### Exercise 2

**Functional updates.**

Starting from the $4\times4$ zero matrix, construct without a loop:

1. the matrix with diagonal $(1,2,3,4)$;
2. a new matrix obtained by adding $-1$ to its last column.

After each operation, check that the preceding matrix has not changed.

In [ ]:
# Complete this section.

## 3. Automatic differentiation

The `jax.grad` transformation takes a scalar-valued function and returns a function computing its gradient. The `argnums` argument specifies the argument with respect to which differentiation is performed. `jax.hessian` gives the Hessian.

JAX transformations are particularly straightforward for pure functions: their result depends only on their arguments and they do not modify external state.

In [ ]:
Q = jnp.array([[3.0, 1.0], [1.0, 2.0]])
b = jnp.array([1.0, -1.0])

def energie(x, Q, b):
    return 0.5 * x @ Q @ x - b @ x

gradient_energie = jax.grad(energie, argnums=0)
hessienne_energie = jax.hessian(energie, argnums=0)

x0 = jnp.array([2.0, -1.0])
g0 = gradient_energie(x0, Q, b)
H0 = hessienne_energie(x0, Q, b)

print(f"gradient = {g0}")
print(f"Hessian =\n{H0}")
assert jnp.allclose(g0, Q @ x0 - b)
assert jnp.allclose(H0, Q)

### Exercise 3

**Directional derivative and Hessian.**

For $p=(1,-2)^\top$, compare $\nabla E(x_0)^\top p$ with

$$
\frac{E(x_0+sp)-E(x_0)}{s}
$$

for $s=10^{-1},10^{-2},\ldots,10^{-8}$. Interpret the initial decrease in the error and its possible subsequent increase. Also check the Hessian from the function `gradient_energie` using `jax.jacfwd`.

In [ ]:
# Complete this section.

### Vector-valued functions

For a vector-valued function, we compute a Jacobian matrix rather than a gradient. JAX provides, in particular, forward-mode `jax.jacfwd` and reverse-mode `jax.jacrev`. They give the same result here, but their costs may differ depending on the input and output dimensions.

In [ ]:
def F(x):
    return jnp.array([x[0] * x[1], jnp.sin(x[0]) + x[1] ** 2])

DF_directe = jax.jacfwd(F)
DF_adjointe = jax.jacrev(F)

print(DF_directe(x0))
assert jnp.allclose(DF_directe(x0), DF_adjointe(x0))

## 4. Structured parameters and `value_and_grad`

The parameters of a machine are rarely gathered into a single vector. JAX handles nested lists, tuples and dictionaries of arrays, called PyTrees. The gradient then has the same structure as the parameters.

The `jax.value_and_grad` transformation computes a function's value and gradient simultaneously.

In [ ]:
def machine_affine(parametres, x):
    return parametres["a"] * x + parametres["b"]

def cout_quadratique(parametres, x, z):
    residu = machine_affine(parametres, x) - z
    return 0.5 * jnp.mean(residu**2)

x_observe = jnp.linspace(-1.0, 1.0, 9)
z_observe = 2.0 * x_observe - 1.0
parametres = {"a": jnp.array(0.0), "b": jnp.array(0.0)}

valeur, gradient = jax.value_and_grad(cout_quadratique)(
    parametres, x_observe, z_observe
)
print(f"objective = {valeur}")
print(f"gradient = {gradient}")

## 5. Vectorisation with `vmap`

The `vmap` transformation automatically applies a function to a batch of arguments. In `in_axes=(None, 0)`, `None` means that the parameters remain fixed and `0` that the first axis of `x` is traversed.

In [ ]:
def machine_scalaire(parametres, x):
    return jnp.tanh(parametres["a"] * x + parametres["b"])

machine_sur_lot = jax.vmap(machine_scalaire, in_axes=(None, 0))
sorties = machine_sur_lot({"a": 2.0, "b": -1.0}, x_observe)
print(sorties)

## 6. Compilation with `jit`

The `jax.jit` transformation asks JAX to compile the computation described by a function for given shapes and types. Compilation occurs on the first call; subsequent calls reuse the compiled computation and may be much faster. Check the function without compilation before applying `jit`.

The notation `@jax.jit`, placed immediately before a function definition, is a Python **decorator**. It is equivalent to `f = jax.jit(f)`.

In [ ]:
def pas_gradient_non_compile(parametres, x, z, alpha):
    valeur, gradient = jax.value_and_grad(cout_quadratique)(parametres, x, z)
    nouveaux_parametres = jax.tree_util.tree_map(
        lambda p, g: p - alpha * g, parametres, gradient
    )
    return nouveaux_parametres, valeur

pas_gradient = jax.jit(pas_gradient_non_compile)
alpha = 0.1
parametres_1, valeur_0 = pas_gradient(
    parametres, x_observe, z_observe, alpha
)
print(parametres_1, valeur_0)

## 7. Random numbers

JAX explicitly represents the state of a random generator with a key. A key must not be reused for two draws: split it with `jax.random.split`. This convention makes randomness compatible with pure functions.

In [ ]:
cle = jax.random.key(2026)
cle_x, cle_bruit = jax.random.split(cle)

x_apprentissage = jax.random.uniform(
    cle_x, shape=(64,), minval=-1.0, maxval=1.0
)
bruit = 0.05 * jax.random.normal(cle_bruit, shape=x_apprentissage.shape)
z_apprentissage = 2.0 * x_apprentissage - 1.0 + bruit

print(x_apprentissage[:5])
print(z_apprentissage[:5])

### Exercise 4

**Identifying an affine machine.**

Starting from `parametres = {"a": 0.0, "b": 0.0}`, apply `pas_gradient` one hundred times to the random observations above.

1. Record the objective value every ten steps.
2. Compare the resulting parameters with the exact values $a=2$ and $b=-1$.
3. Repeat with several noise amplitudes.
4. What happens if the target becomes $z=x^2-1$? Is this a limitation of the algorithm or of the chosen machine class?

In [ ]:
# Complete this section.

## Summary

A JAX transformation takes a function and produces a new function: `grad` differentiates, `vmap` vectorises and `jit` compiles. Combining them will allow us to define and fit more complex machines without writing their derivatives by hand.